# Amaliyot 15 — Agent va ReAct sikli

**Mos ma'ruza:** L15 — Sun'iy intellekt agentlarini yaratish  
**Capstone natijasi:** `DocumentAssistantAgent`  
**Muhit:** asosiy qism CPU'da ishlaydi; lokal Qwen bo'limi uchun Kaggle GPU va Internet kerak. API kaliti kerak emas.

Bugun tayyor agent kodini ko'chirmaymiz. Uni bosqichma-bosqich quramiz:

1. kichik NLP vositalari;
2. vosita reyestri;
3. qoidaviy rejalashtiruvchi;
4. ReAct boshqaruv sikli;
5. guardrails va capstone klassi.

**Taxminiy vaqt:** kirish 6 min · toollar 12 min · planner 12 min · ReAct 18 min · capstone 12 min · lokal Qwen 15 min · yakun 5 min.

## 0. Muhit

Asosiy lab faqat Python standart kutubxonasidan foydalanadi. Shu sababli Kaggle'da paket o'rnatish yoki internetni yoqish shart emas.

In [16]:
import json
import platform
import re
from typing import Optional

print("Python:", platform.python_version())
print("Asosiy qism tayyor. Lokal Qwen bo'limida GPU kerak bo'ladi.")

Python: 3.12.13
Asosiy qism tayyor. Lokal Qwen bo'limida GPU kerak bo'ladi.


## 1. Agent nimani boshqaradi?

Oddiy pipeline qadamlari oldindan belgilangan. Agent esa so'rov va oldingi kuzatuvlarga qarab keyingi vositani tanlaydi.

Agentning boshqaruv sikli

Bu notebookda haqiqiy LLM o'rniga tushunarli **qoidaviy planner** ishlatamiz. Shunda agent siklining mexanikasi API kalitisiz ko'rinadi. Oxirida planner qismini lokal Qwen3 modeli bilan almashtiramiz.

## 2. Agent ishlata oladigan vositalar

Tool — agent chaqira oladigan, bitta aniq vazifani bajaruvchi funksiya. Avval uchta kichik backend tayyorlaymiz: sentiment, qidiruv va qisqartirish.

Tokenizatsiya yordamchi qadamdir; u agent mavzusi emas, shuning uchun juda sodda qoldirilgan.

In [17]:
def tokenize(text: str) -> list[str]:
    """O'zbekcha apostroflarni saqlagan holda sodda tokenizatsiya."""
    return re.findall(r"[a-zA-Z0-9_ʻʼ’'-]+", text.lower())

### 2.1 Sentiment vositasi

Bu yerda maqsad kuchli klassifikator yaratish emas. Agent chaqira oladigan, kirish va chiqishi aniq funksiyani olish kifoya. Keyinchalik shu funksiyaning o'rniga oldingi capstone klassifikatorini ulash mumkin.

In [18]:
def sentiment_classify(text: str) -> dict:
    """Kichik demo sentiment vositasi."""
    tokens = set(tokenize(text))
    positive_words = {"yaxshi", "ajoyib", "zo'r", "foydali", "qulay"}
    negative_words = {"yomon", "qiyin", "sifatsiz", "muammo", "sekin"}

    positive_count = len(tokens & positive_words)
    negative_count = len(tokens & negative_words)
    if positive_count > negative_count:
        return {"label": "ijobiy", "confidence": 0.82}
    if negative_count > positive_count:
        return {"label": "salbiy", "confidence": 0.79}
    return {"label": "neytral", "confidence": 0.55}

### 2.2 Kichik bilimlar bazasi

Alohida dataset yuklamaymiz. Beshta yozuv agent va qidiruv oqimini ko'rish uchun yetarli.

In [19]:
KNOWLEDGE_BASE = [
    {"title": "NLP", "text": "NLP kompyuterga inson tilidagi matnni tahlil qilishga yordam beradi."},
    {"title": "Transformer", "text": "Transformer attention yordamida tokenlar orasidagi bog'lanishni modellashtiradi."},
    {"title": "RAG", "text": "RAG avval hujjat topadi, keyin savol va kontekstni generatorga yuboradi."},
    {"title": "Agent", "text": "Agent maqsadga qarab vosita tanlaydi, natijani kuzatadi va keyingi qadamni belgilaydi."},
    {"title": "O'zbek NLP", "text": "O'zbek tili agglutinativ bo'lgani uchun qo'shimchalar matn tahlilida muhim."},
]

`retrieve_docs()` so'rov va hujjat orasidagi umumiy tokenlarni sanaydi. Bu RAG modelining o'rnini bosmaydi; u faqat agent uchun tez va deterministik qidiruv toolidir.

In [20]:
def retrieve_docs(query: str) -> dict:
    """So'rovga eng ko'p umumiy tokeni bor hujjatni topadi."""
    query_tokens = set(tokenize(query))
    scored_documents = []
    for document in KNOWLEDGE_BASE:
        document_tokens = set(tokenize(document["text"] + " " + document["title"]))
        score = len(query_tokens & document_tokens)
        scored_documents.append((score, document))

    best_score, best_document = max(scored_documents, key=lambda item: item[0])
    return {
        "title": best_document["title"],
        "text": best_document["text"],
        "score": best_score,
    }

### 2.3 Qisqartirish vositasi

Bu ekstraktiv demo birinchi ikki gapni oladi. Agent nuqtayi nazaridan muhim narsa: funksiya matn qabul qiladi va strukturali natija qaytaradi.

In [21]:
def summarize(text: str) -> dict:
    """Birinchi ikki gapni qaytaruvchi ekstraktiv demo vosita."""
    sentences = [part.strip() for part in re.split(r"[.!?]+", text) if part.strip()]
    selected = sentences[:2]
    summary = ". ".join(selected) + ("." if selected else "")
    return {"summary": summary, "sentence_count": len(selected)}

Uchala backendni alohida sinab ko'ring. Agentni yozishdan oldin toolning o'zi ishlashiga ishonch hosil qilish debuggingni osonlashtiradi.

In [22]:
print(sentiment_classify("Bu xizmat juda yaxshi va qulay."))
print(retrieve_docs("RAG nima?"))
print(summarize("Agent vosita tanlaydi. Natijani kuzatadi. So'ng javob beradi."))

{'label': 'ijobiy', 'confidence': 0.82}
{'title': 'RAG', 'text': 'RAG avval hujjat topadi, keyin savol va kontekstni generatorga yuboradi.', 'score': 1}
{'summary': 'Agent vosita tanlaydi. Natijani kuzatadi.', 'sentence_count': 2}


## 3. Tool reyestri

Agentga faqat Python funksiyasini berish yetmaydi. Vositaning nomi, vazifasi va kutilgan kirishi ham tushunarli bo'lishi kerak.

Tool tuzilishi

Bizning qoidaviy plannerimiz `keywords` maydonidan foydalanadi. Haqiqiy LLM esa odatda `description` va argument sxemasidan ma'no chiqaradi.

In [23]:
TOOLS = {
    "sentiment_classify": {
        "function": sentiment_classify,
        "description": "Matn hissiyotini aniqlaydi.",
        "keywords": ["hissiyot", "ijobiy", "salbiy", "yaxshi", "yomon", "baho"],
    },
    "retrieve_docs": {
        "function": retrieve_docs,
        "description": "Bilimlar bazasidan mos hujjatni topadi.",
        "keywords": ["top", "qidir", "ma'lumot", "hujjat", "nima", "haqida"],
    },
    "summarize": {
        "function": summarize,
        "description": "Berilgan matnni qisqartiradi.",
        "keywords": ["xulosa", "qisqa", "qisqartir", "umumlashtir"],
    },
}

Reyestrni ko'zdan kechiring. Har bir tool bir xil shartnomaga ega: `function`, `description`, `keywords`.

In [24]:
for tool_name, tool_info in TOOLS.items():
    print(f"{tool_name:20} -> {tool_info['description']}")

sentiment_classify   -> Matn hissiyotini aniqlaydi.
retrieve_docs        -> Bilimlar bazasidan mos hujjatni topadi.
summarize            -> Berilgan matnni qisqartiradi.


## 4. Mashq 1 — keyingi vositani tanlash

Ma'ruzadagi formula:

$$a_t = \operatorname{LLM}(q, h_{t-1})$$

Biz LLM o'rniga `select_tool()` qoidaviy plannerini yozamiz. U so'rovdagi kalit so'zlarni sanaydi va tarixda ishlatilgan toolni qayta tanlamaydi.

In [25]:
def select_tool(query: str, history: list[dict], tools: dict) -> Optional[str]:
    """LLM o'rnidagi tushunarli, qoidaviy rejalashtiruvchi."""
    # SIZNING KODINGIZ:
    # 1. query ni kichik harfga o'tkazing.
    query_lower = query.lower()
    # 2. history ichidagi ishlatilgan tool nomlarini yig'ing.
    used_tools = {record['tool'] for record in history}
    # 3. Qolgan toollar uchun keyword mosliklarini hisoblang.
    scores = {}
    for tool_name, tool_info in tools.items():
        if tool_name in used_tools:
            continue
        scores[tool_name] = sum(
            keyword in query_lower for keyword in tool_info["keywords"]
        )
    # 4. Eng katta ball 0 bo'lsa None, aks holda tool nomini qaytaring.
    if not scores or max(scores.values())==0:
        return None
    return max(scores, key = scores.get)
    # raise NotImplementedError("select_tool() ni to'ldiring")

Tekshiruvlar uch holatni qamrab oladi: sentiment tanlash, qidiruv tanlash va tarixdagi toolni takrorlamaslik.

In [26]:
assert select_tool("Bu mahsulot yaxshi", [], TOOLS) == "sentiment_classify"
assert select_tool("RAG haqida ma'lumot top", [], TOOLS) == "retrieve_docs"

used_history = [{"tool": "sentiment_classify"}]
assert select_tool("Bu mahsulot yaxshi", used_history, TOOLS) is None
print("Mashq 1: PASS")

Mashq 1: PASS


## 5. Mashq 2 — harakatni bajarish

Planner faqat tool nomini qaytaradi. Executor shu nomni tekshiradi va mos Python funksiyasini chaqiradi:

$$o_t = \operatorname{tool}(a_t)$$

Noma'lum nomni jimgina o'tkazib yubormang; tushunarli xato chiqaring.

In [27]:
def execute_action(tool_name: str, query: str, tools: dict) -> dict:
    """Tanlangan vositani xavfsiz tekshiradi va bajaradi."""
    # SIZNING KODINGIZ:
    # Noma'lum tool uchun ValueError chiqaring, aks holda funksiyani chaqiring.
    if tool_name not in tools:
        raise ValueError(f"Noma'lum vosita: {tool_name}")
        
    tool_function = tools[tool_name]["function"]
    return tool_function(query)
    # raise NotImplementedError("execute_action() ni to'ldiring")

Quyidagi tekshiruv haqiqiy natijani va noto'g'ri tool uchun guardrailni sinaydi.

In [28]:
observation = execute_action("sentiment_classify", "Xizmat yaxshi", TOOLS)
assert observation["label"] == "ijobiy"

try:
    execute_action("delete_everything", "test", TOOLS)
except ValueError as error:
    print("Guardrail ishladi:", error)

Guardrail ishladi: Noma'lum vosita: delete_everything


## 6. Mashq 3 — ReAct sikli

Har qadamda planner tool tanlaydi, executor uni bajaradi va natija tarixga yoziladi:

$$a_t = \operatorname{planner}(q,h_{t-1}), \qquad o_t = \operatorname{tool}(a_t)$$
$$h_t = h_{t-1} \;\Vert\; [(a_t,o_t)]$$

ReAct tarixining o'sishi

In [29]:
def format_final_answer(history: list[dict]) -> str:
    """Kuzatuvlarni foydalanuvchiga ko'rinadigan qisqa javobga aylantiradi."""
    if not history:
        return "Mos vosita topilmadi. So'rovni aniqroq yozing."
    parts = []
    for record in history:
        observation = record["observation"]
        parts.append(f"{record['tool']}: {json.dumps(observation, ensure_ascii=False)}")
    return " | ".join(parts)

`run_react()` uchta guardrailga ega bo'lishi kerak:

- `max_steps` cheksiz siklni cheklaydi;
- `None` yangi mos tool yo'qligini bildiradi;
- planner tarixdagi toolni qayta tanlamaydi.

In [30]:
def run_react(
    query: str, tools: dict, max_steps: int = 3,
    planner=select_tool,
) -> list[dict]:
    """Harakat va kuzatuvlardan iborat cheklangan ReAct sikli."""
    history = []
    # SIZNING KODINGIZ:
    # max_steps marta planner bilan tool tanlang, None bo'lsa to'xtang,
    for step_number in range(1, max_steps+1):
        tool_name = planner(query, history, tools)
        if tool_name is None:
            break
        observation = execute_action(tool_name, query, tools)
        history.append({
            'step': step_number,
            "tool": tool_name,
            "input": observation,
            "observation": observation
        })
    return history
    # toolni bajaring va step/tool/input/observation ni history ga qo'shing.
    # raise NotImplementedError("run_react() ni to'ldiring")

Endi bir so'rovda ikki xil niyatni yozamiz. Natijada ikki xil tool ishlashi va tarixda ikkita kuzatuv paydo bo'lishi kerak.

In [31]:
query = "Bu kurs yaxshi. RAG haqida ma'lumot ham top."
trace = run_react(query, TOOLS, max_steps=3)

assert [record["tool"] for record in trace] == [
    "retrieve_docs", "sentiment_classify"
]
print(format_final_answer(trace))

retrieve_docs: {"title": "RAG", "text": "RAG avval hujjat topadi, keyin savol va kontekstni generatorga yuboradi.", "score": 1} | sentiment_classify: {"label": "ijobiy", "confidence": 0.82}


Trace agentning kuzatiladigan ish jurnalidir. U yashirin ichki mulohazani emas, bajarilgan harakat va qaytgan kuzatuvni saqlaydi.

In [32]:
for record in trace:
    print(f"{record['step']}-qadam")
    print("  harakat :", record["tool"])
    print("  kuzatuv :", record["observation"])

1-qadam
  harakat : retrieve_docs
  kuzatuv : {'title': 'RAG', 'text': 'RAG avval hujjat topadi, keyin savol va kontekstni generatorga yuboradi.', 'score': 1}
2-qadam
  harakat : sentiment_classify
  kuzatuv : {'label': 'ijobiy', 'confidence': 0.82}


## 7. Guardrail tajribasi

Murakkab so'rov uchta toolga mos tushadi. `max_steps=2` bo'lsa, agent uchinchi harakatga o'tmasligi kerak. Bu limit sifat va xarajat orasidagi boshqariladigan kompromissdir.

In [33]:
complex_query = "Bu yaxshi matn. RAG haqida ma'lumot top va qisqa xulosa qil."
limited_trace = run_react(complex_query, TOOLS, max_steps=2)

assert len(limited_trace) == 2
print("Bajarilgan toollar:", [record["tool"] for record in limited_trace])

Bajarilgan toollar: ['retrieve_docs', 'summarize']


### Qoidaviy plannerning chegarasi

Kalit so'z ishlatilmagan parafrazda planner niyatni tushunmasligi mumkin. Bu tajriba qoidaviy routing va LLM routing orasidagi farqni ko'rsatadi.

In [34]:
paraphrase = "Ushbu yozuvning asosiy fikrlarini qisqa bayon et."
selected = select_tool(paraphrase, [], TOOLS)
print("Tanlangan tool:", selected)
print("Nega bu natija chiqdi? Qaysi tavsif yoki keywordni yaxshilash mumkin?")

Tanlangan tool: summarize
Nega bu natija chiqdi? Qaysi tavsif yoki keywordni yaxshilash mumkin?


## 8. Yakuniy vazifa — capstone klassi

Endi oldingi funksiyalarni bitta interfeysga yig'amiz:

```text
agent = DocumentAssistantAgent()
answer = agent.run(user_message)
trace = agent.last_trace()
```

Klass tool funksiyalarini qayta implementatsiya qilmaydi. U reyestrni qabul qiladi va bugun yozilgan `run_react()` dan foydalanadi. Keyinchalik `TOOLS` ichidagi demo funksiyalar o'rniga oldingi capstone modullarining metodlarini berish mumkin.

In [35]:
class DocumentAssistantAgent:
    """P15 capstone natijasi: planner va toollari almashtiriladigan agent."""

    def __init__(
        self, tools: Optional[dict] = None, max_steps: int = 3,
        planner=select_tool,
    ):
        self.tools = tools or TOOLS
        self.max_steps = max_steps
        self.planner = planner
        self.history = []

    def run(self, user_message: str) -> str:
        self.history = run_react(
            query=user_message,
            tools=self.tools,
            max_steps=self.max_steps,
            planner=self.planner,
        )

        if not self.history:
            return ""
        
        return str(self.history[-1]["observation"])


    def last_trace(self) -> list[dict]:
        # SIZNING KODINGIZ: tarixning nusxasini qaytaring.
        return self.history.copy()

    def reset(self) -> None:
        self.history = []

    def tool_names(self) -> list[str]:
        return list(self.tools)

In [36]:
agent = DocumentAssistantAgent()

answer = agent.run("Bu kurs yaxshi. Agent haqida ma'lumot top.")

print(answer)
print(agent.last_trace())

{'label': 'ijobiy', 'confidence': 0.82}
[{'step': 1, 'tool': 'retrieve_docs', 'input': {'title': 'Agent', 'text': 'Agent maqsadga qarab vosita tanlaydi, natijani kuzatadi va keyingi qadamni belgilaydi.', 'score': 1}, 'observation': {'title': 'Agent', 'text': 'Agent maqsadga qarab vosita tanlaydi, natijani kuzatadi va keyingi qadamni belgilaydi.', 'score': 1}}, {'step': 2, 'tool': 'sentiment_classify', 'input': {'label': 'ijobiy', 'confidence': 0.82}, 'observation': {'label': 'ijobiy', 'confidence': 0.82}}]


Shartnoma tekshiruvi capstone klassining tashqi xulqini tekshiradi. Bu katakdan o'tsa, P15 natijasi tayyor hisoblanadi.

In [37]:
agent = DocumentAssistantAgent(max_steps=3)
answer = agent.run("Bu kurs yaxshi. Agent haqida ma'lumot top.")

assert isinstance(answer, str) and answer
assert len(agent.last_trace()) == 2
assert agent.tool_names() == list(TOOLS)
agent.reset()
assert agent.last_trace() == []
print("Capstone shartnomasi: PASS")

Capstone shartnomasi: PASS


## 9. Local LLM — Qwen3 planner

Endi faqat planner qismini lokal LLM bilan almashtiramiz. Controller, toollar va guardrails o'zgarmaydi:

```text
qoidaviy planner ─┐
                  ├─→ bir xil ReAct controller → bir xil toollar
Qwen3 planner ────┘
```

`Qwen/Qwen3-1.7B` Kaggle GPU'da lokal ishlaydi. API kaliti kerak emas, lekin birinchi yuklash uchun **Internet** va GPU kerak. Kaggle paketlari eski bo'lsa, notebook boshida faqat quyidagini o'rnating; `torch`ni qayta o'rnatmang:

```text
%pip install -q -U "transformers>=4.51" accelerate
```

**Dars uchun:** model cache'ga tushishi uchun o'qituvchi bu bo'limni mashg'ulotdan oldin bir marta ishga tushirishi ma'qul.

In [38]:
%pip install -q -U "transformers>=4.51" accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 82.0 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 19.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [39]:
USE_LOCAL_LLM = True
LOCAL_MODEL_ID = "Qwen/Qwen3-1.7B"

print("Local LLM:", "yoqilgan" if USE_LOCAL_LLM else "o'chirilgan")
print("Model:", LOCAL_MODEL_ID)

Local LLM: yoqilgan
Model: Qwen/Qwen3-1.7B


### 9.1 Modelni bir marta yuklash

Model har bir so'rovda qayta yuklanmaydi. `float16` 1.7B modelni Kaggle T4 xotirasiga joylashtirishga yordam beradi. `device_map="auto"` mavjud GPU'ni tanlaydi.

In [40]:
if USE_LOCAL_LLM:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    assert torch.cuda.is_available(), "Kaggle Settings → Accelerator → GPU ni yoqing."
    local_tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_ID)
    local_model = AutoModelForCausalLM.from_pretrained(
        LOCAL_MODEL_ID,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    local_model.eval()
    print("GPU:", torch.cuda.get_device_name(0))

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


GPU: Tesla T4


### 9.2 Python reyestridan tool schema yaratish

Qwen tool nomi, tavsifi va argument sxemasini ko'radi. Barcha demo toollar bitta matnli `input` qabul qiladigan umumiy adapter sifatida taqdim etiladi.

In [41]:
def build_tool_schemas(tools: dict) -> list[dict]:
    schemas = []
    for tool_name, tool_info in tools.items():
        schemas.append({
            "type": "function",
            "function": {
                "name": tool_name,
                "description": tool_info["description"],
                "parameters": {
                    "type": "object",
                    "properties": {"input": {"type": "string"}},
                    "required": ["input"],
                },
            },
        })
    return schemas

### 9.3 Strukturali chiqishni tekshirish

Lokal model ham ba'zan formatni buzishi mumkin. Parser javob ichidan birinchi JSON obyektini oladi; noma'lum yoki takroriy toolni guardrail rad etadi.

In [42]:
def first_json_object(text: str) -> dict:
    decoder = json.JSONDecoder()
    for position, character in enumerate(text):
        if character != "{":
            continue
        try:
            value, _ = decoder.raw_decode(text[position:])
            if isinstance(value, dict):
                return value
        except json.JSONDecodeError:
            continue
    raise ValueError(f"Qwen JSON qaytarmadi: {text[:160]}")

def validate_tool_call(text: str, history: list[dict], tools: dict):
    if "FINISH" in text.upper() and "{" not in text:
        return None
    action = first_json_object(text)
    tool_name = action.get("name")
    used_tools = {record["tool"] for record in history}
    if tool_name not in tools or tool_name in used_tools:
        raise ValueError(f"Noto'g'ri yoki takroriy tool: {tool_name}")
    return tool_name

### 9.4 Qwen bilan matn generatsiyasi

`enable_thinking=False` tool tanlashni qisqa va tez qiladi. Modeldan yashirin mulohaza emas, tool-call formatidagi qisqa javob so'raladi.

In [43]:
def generate_qwen_action(messages: list[dict], tools: dict) -> str:
    prompt = local_tokenizer.apply_chat_template(
        messages, tools=build_tool_schemas(tools), tokenize=False,
        add_generation_prompt=True, enable_thinking=False,
    )
    model_inputs = local_tokenizer(prompt, return_tensors="pt").to(local_model.device)
    with torch.inference_mode():
        output_ids = local_model.generate(
            **model_inputs, max_new_tokens=96, do_sample=False
        )
    new_ids = output_ids[0, model_inputs["input_ids"].shape[1]:]
    generated_text = local_tokenizer.decode(new_ids, skip_special_tokens=True)
    print("Qwen chiqishi:", generated_text.strip())
    return generated_text

### 9.5 Qwen planner

Planner so'rov va oldingi kuzatuvlarni modelga beradi. So'ng strukturali chiqishni tekshirib, controller kutadigan tool nomini qaytaradi.

In [44]:
def qwen_select_tool(query: str, history: list[dict], tools: dict):
    history_view = [
        {"tool": item["tool"], "observation": item["observation"]}
        for item in history
    ]
    messages = [
        {"role": "system", "content": (
            "Siz agent plannersiz. Bitta kerakli va ishlatilmagan toolni chaqiring. "
            "Boshqa tool kerak bo'lmasa faqat FINISH yozing."
        )},
        {"role": "user", "content": (
            f"So'rov: {query}\nOldingi kuzatuvlar: "
            f"{json.dumps(history_view, ensure_ascii=False)}"
        )},
    ]
    generated_text = generate_qwen_action(messages, tools)
    try:
        return validate_tool_call(generated_text, history, tools)
    except ValueError as error:
        print("Planner guardrail agentni to'xtatdi:", error)
        return None

### 9.6 Bir xil agent, boshqa planner

Endi capstone klassiga faqat planner funksiyasini beramiz. Qwen toolni tanlaydi; mavjud controller uni bajaradi, kuzatuvni tarixga qo'shadi va keyingi qadamni so'raydi.

In [45]:
if USE_LOCAL_LLM:
    qwen_agent = DocumentAssistantAgent(
        planner=qwen_select_tool,
        max_steps=3,
    )
    qwen_answer = qwen_agent.run(
        "Bu kurs menga manzur bo'ldi. RAG mohiyatini ham tushuntiring."
    )
    print("\nYakuniy javob:", qwen_answer)
    print("Qadamlar:", [item["tool"] for item in qwen_agent.last_trace()])

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Qwen chiqishi: <tool_call>
{"name": "summarize", "arguments": {"input": "Bu kurs menga manzur bo'ldi. RAG mohiyatini ham tushuntiring."}}
</tool_call>
Qwen chiqishi: <tool_call>
{"name": "sentiment_classify", "arguments": {"input": "Bu kurs menga manzur bo'ldi. RAG mohiyatini ham tushuntiring."}}
</tool_call>
Qwen chiqishi: <tool_call>
{"name": "summarize", "arguments": {"input": "Bu kurs menga manzur bo'ldi. RAG mohiyatini ham tushuntiring."}}
</tool_call>
Planner guardrail agentni to'xtatdi: Noto'g'ri yoki takroriy tool: summarize

Yakuniy javob: {'label': 'neytral', 'confidence': 0.55}
Qadamlar: ['summarize', 'sentiment_classify']


**Taqqoslash:** qoidaviy planner faqat yozilgan keywordlarni ko'radi; Qwen esa parafraz ma'nosidan tool tanlashi mumkin. Lekin Qwen sekinroq va ba'zan noto'g'ri format qaytaradi — shu sabab parser, allowlist va `max_steps` saqlanib qoladi.

LangChain ham xuddi shu qismlarni framework sifatida boshqaradi. Bu labda lokal modelning haqiqiy tool-call formati ko'rinishi uchun `transformers` bilan bevosita ishladik.

## 10. Yakun va chiqish savollari

Bugun qurilgan zanjir:

```text
so'rov → planner → tool → observation → history → yakuniy javob
```

Muhokama qiling:

1. Nega `max_steps` agent uchun xavfsizlik chegarasi hisoblanadi?
2. Qoidaviy planner qaysi parafrazlarda xato qiladi?
3. Haqiqiy LLM tool tanlashda `description` va argument sxemasidan qanday foydalanadi?
4. `DocumentAssistantAgent` ga oldingi RAG yoki sentiment capstone modulini qanday ulardingiz?

**Chiqish bileti:** agentning oddiy pipeline'dan eng muhim farqini bir jumlada yozing.